# Hands-on Projects (VLMs)

**Module:** 15 — VLMs & Multimodal

Receipt extraction, slide QA, alt-text, and a bakeoff — with acceptance tests.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Implement four scoped multimodal projects with acceptance criteria
- Use schemas, validators, and mock VLM responses
- Ship diagrams, tests, failure modes, and cost sketches


## Project 1 — Receipt Extractor

**Brief:** Receipt image → JSON `{merchant, date, total, currency, line_items[]}`.

```mermaid
flowchart LR
  R[Image] --> P[Preprocess] --> V[VLM JSON]
  P --> O[OCR]
  V --> C[Reconcile] --> Q{valid?}
  O --> C
  Q -->|no| H[HITL]
  Q -->|yes| S[Store]
```

| ID | Acceptance |
|----|------------|
| R1 | Missing total fails schema |
| R2 | OCR/VLM disagree >1% → HITL |
| R3 | Provenance includes crop id |


In [ ]:
# Project 1 starter
import re
from typing import Any

def parse_money(s: str):
    m = re.search(r"(\d+\.\d{2})", s)
    return float(m.group(1)) if m else None

def validate_receipt(fields: dict[str, Any]):
    return [f"missing:{k}" for k in ("merchant","date","total","currency") if fields.get(k) in (None,"")]

def reconcile(vlm: dict, ocr_text: str):
    ocr_total = parse_money(ocr_text)
    status = "HITL" if ocr_total is not None and abs(ocr_total-float(vlm["total"]))>0.01 else "ok"
    return {"fields": vlm, "status": status, "ocr_total": ocr_total, "provenance": {"crop":"full_page"}}

vlm={"merchant":"Cafe","date":"2026-07-01","total":12.5,"currency":"USD","line_items":[{"desc":"Latte","price":4.5}]}
print(validate_receipt(vlm), reconcile(vlm,"TOTAL 12.50")["status"], reconcile(vlm,"TOTAL 15.00")["status"])


In [ ]:
# Project 1 tests
assert validate_receipt({"merchant":"x"}) != []
assert reconcile({"merchant":"c","date":"d","total":1.0,"currency":"USD","line_items":[]},"TOTAL 9.00")["status"]=="HITL"
print("P1 ok")


### Try it yourself — Receipt extractor

1. Add tax/subtotal math checks.
2. Support EUR comma decimals.
3. Assert R1–R3 in a test function.


## Project 2 — Slide Deck QA

**Brief:** Slide images (+ notes) → `{answer, page_id, confidence}`; UNKNOWN if missing.

| ID | Acceptance |
|----|------------|
| S1 | ≤3 pages retrieved |
| S2 | page_id always present when answered |
| S3 | UNKNOWN without evidence |


In [ ]:
# Project 2 starter
SLIDES=[
    {"page_id":"3","text":"Roadmap: ship multimodal RAG in Q3","uri":"slide3.png"},
    {"page_id":"7","text":"Pricing starts at $20/seat","uri":"slide7.png"},
]
def retrieve_slides(q,k=3):
    qw=set(q.lower().split())
    scored=sorted(((len(qw & set(s["text"].lower().split())),s) for s in SLIDES), reverse=True)
    return [s for sc,s in scored[:k] if sc>0]

def answer(q):
    hits=retrieve_slides(q)
    if not hits: return {"answer":"UNKNOWN","page_id":None,"confidence":0.0}
    return {"answer":hits[0]["text"],"page_id":hits[0]["page_id"],"confidence":0.7,"pages_used":[h["page_id"] for h in hits]}
print(answer("When do we ship multimodal RAG?"))
print(answer("Who is the CEO?"))


In [ ]:
# Project 2 tests
a=answer("multimodal RAG")
assert a["page_id"] is not None and len(a.get("pages_used", [])) <= 3
assert answer("Who is the CEO?")["answer"] == "UNKNOWN"
print("P2 ok")


### Try it yourself — Slide QA

1. Require citations in a response schema.
2. Add notes field to slides and search it.
3. Measure recall@3 on 10 frozen questions.


## Project 3 — Accessibility Alt-Text Service

**Brief:** API returns concise alt-text; avoids guessing names; rate-limits; stores hash not raw bytes by default.

| ID | Acceptance |
|----|------------|
| A1 | Alt-text <= 140 chars |
| A2 | Proper-name heuristic softened to "a person" |
| A3 | Audit log has image_hash only |


In [ ]:
# Project 3 starter
import hashlib, re

def alt_text_service(image_bytes: bytes, model_caption: str) -> dict:
    caption = re.sub(r"\b[A-Z][a-z]+ [A-Z][a-z]+\b", "a person", model_caption)
    if len(caption) > 140:
        caption = caption[:137] + "..."
    return {"alt_text": caption, "image_hash": hashlib.sha256(image_bytes).hexdigest()[:16], "store_raw": False}

print(alt_text_service(b"img", "John Smith riding a red bicycle near downtown"))
assert alt_text_service(b"x", "A")["store_raw"] is False
print("P3 ok")


### Try it yourself — Alt-text service

1. Add per-user rate limiting from Module 08 patterns.
2. Refuse medical diagnosis style prompts.
3. Strip GPS-like metadata keys before logging.


## Project 4 — Bakeoff Notebook

**Brief:** Freeze labeled images; compare mini vs frontier (mocked OK); print metrics; pick winner under budget.

| ID | Acceptance |
|----|------------|
| B1 | Frozen dataset list |
| B2 | Metrics table printed |
| B3 | Recommendation under $/1K budget |


In [ ]:
# Project 4 starter
from statistics import mean

data = [
    {"id": "i1", "gold": "12.50", "mini": "12.50", "frontier": "12.50"},
    {"id": "i2", "gold": "cafe mocha", "mini": "coffee", "frontier": "cafe mocha"},
    {"id": "i3", "gold": "2", "mini": "2", "frontier": "2"},
]

def acc(model: str) -> float:
    return mean(row[model].lower() == row["gold"].lower() for row in data)

costs = {"mini": 0.40, "frontier": 3.50}
budget = 1.00
candidates = [m for m, c in costs.items() if c <= budget]
winner = max(candidates, key=acc) if candidates else "none"
print({"mini": acc("mini"), "frontier": acc("frontier"), "candidates": candidates, "winner": winner})
print("P4 ok")


## Deliverables Checklist

- [ ] Architecture diagram per project
- [ ] Schema + validators
- [ ] >=5 automated asserts
- [ ] Failure mode list (hallucinated totals, bad crops, ACL)
- [ ] Cost sketch ($/1K images)
- [ ] Safety/retention note for stored images


In [ ]:
# Shared smoke tests
assert validate_receipt({"merchant": "x"}) != []
assert answer("CEO name")["answer"] == "UNKNOWN"
assert alt_text_service(b"x", "A")["store_raw"] is False
assert winner in costs
print("all project smokes passed")


### Try it yourself — Ship projects

1. Add streaming page batching for a 50-slide deck.
2. Log structured audit JSON for receipt HITL events.
3. Expand bakeoff with soft-match and CER for numeric fields.

**Stretch:** Wire one project to a live API using YOUR_OPENAI_API_KEY from the environment.


## Knowledge Check

**Q1.** Why reconcile OCR with VLM on receipts?

<details><summary>Answer</summary>

Disagreement is a cheap signal that catches fluent but wrong totals before money moves.

</details>

**Q2.** What belongs in alt-text audit logs?

<details><summary>Answer</summary>

Hashes, model id, timestamps — not raw image bytes by default.

</details>


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `bakeoff` | Fixed eval comparing models |
| `provenance` | Origin metadata for a field |
| `alt-text` | Accessibility description |
| `reconcile` | Resolve OCR vs VLM disagreements |
| `acceptance` | Binary project requirement |


## Key Takeaways

- Projects should exercise extract -> validate -> HITL
- Citations and provenance turn demos into systems
- Bakeoffs under budget mirror real selection
- Tests and failure modes beat screenshots


## Production Incident Patterns — VLM projects

| Symptom | Likely cause | First fix |
|---------|--------------|-----------|
| Sudden cost spike | `detail=high` on huge pages | Resize + tile budget |
| Fluent wrong fields | VLM hallucination | OCR hybrid + schema |
| Cross-customer leak | Missing tenant filter | ACL in retriever code |
| Flaky eval scores | Unfrozen prompts/models | Pin versions + bakeoff set |
| Latency SLO burn | Full-page high detail | Crop ROI → mini model |

```
ASCII control loop:
  ingest -> normalize -> route model -> generate -> validate -> (HITL|export)
                     ^                              |
                     +-------- metrics/audit <------+
```


In [ ]:
# Cross-cutting: redact secrets before logging multimodal payloads
import re, json

SECRET_RE = re.compile(r"(api[_-]?key|bearer\s+[A-Za-z0-9._\-]+)", re.I)

def safe_log(payload: dict) -> str:
    s = json.dumps(payload)
    s = SECRET_RE.sub("***", s)
    if "base64," in s:
        s = re.sub(r"base64,[A-Za-z0-9+/=]+", "base64,[REDACTED]", s)
    return s[:500]

print(safe_log({
    "model": "gpt-4o",
    "api_key": "YOUR_OPENAI_API_KEY",
    "content": "data:image/png;base64,AAAABBBBCCCC",
    "topic": "VLM projects",
}))


## Mini Case Study — VLM projects

**Scenario:** A team ships a vision feature in one week. Demo looks great on three happy-path images.
**Week 2:** finance reports wrong totals; legal asks about image retention; GPU/API bill 4× forecast.

**Retro questions**
1. What was the output contract (schema) on day one?
2. Which failure mode had no metric?
3. Was there a crop/detail budget?
4. Who owns HITL and appeals?

**Design rule:** if a field can move money or identity, it needs a validator + disagreement path before automation.


In [ ]:
# Cross-cutting: simple SLO helper for vision endpoints
from dataclasses import dataclass

@dataclass
class VisionSLO:
    availability: float = 0.995
    p95_ms: int = 4000
    max_critical_field_error_rate: float = 0.005

def breached(slo: VisionSLO, avail: float, p95: int, crit_err: float) -> list[str]:
    out = []
    if avail < slo.availability: out.append("availability")
    if p95 > slo.p95_ms: out.append("latency")
    if crit_err > slo.max_critical_field_error_rate: out.append("critical_accuracy")
    return out or ["ok"]

print("VLM projects", breached(VisionSLO(), 0.99, 5200, 0.02))


### Try it yourself — VLM projects ops

1. Write a one-page runbook section for on-call when VLM projects critical_accuracy SLO breaches.
2. Add a dashboard sketch: cost/1k images, CER/field error, HITL rate, p95 latency.
